# Graph build — Layer A + Layer B

Clean, linear build of the FoodScholar knowledge graph:

| # | Phase | Call |
|---|-------|------|
| 1 | Configure | `FoodScholar.from_config(...)` |
| 2 | Load corpus + annotations | `load_chunks` (offline) or `ingest` (Elastic) |
| 3 | Embed chunks | `fs.embed()` |
| 4 | Build entities | `fs.build_entities()` |
| 5 | **Layer A** (FoodOn projection + aliasing) | `fs.build_layer_a()` |
| 6 | Attach chunks to shelves | `fs.attach()` |
| 7 | **Layer B** (themes) | `fs.build_layer_b()` |
| 8 | **Interactive tree** | `fs.viz.layer_a_tree(...).render("tree")` |

Layer A uses the 1a+ backbone projection (umbrella rule + single-parent + single-child
collapse) and then an LLM **aliasing** pass that gives jargon shelves a friendly
`display_label` — additive, ids/structure untouched. Set `BACKEND='memory'` to run
fully offline from `data/annotated.parquet` (build it once with
`scripts/make_annotated_parquet.py`); set `BACKEND='elastic'` for the real stores.

## 1. Configure

In [ ]:
import os
from pathlib import Path

from foodscholar import FoodScholar
from foodscholar.config import FoodScholarConfig
from foodscholar.logging import configure_logging

configure_logging(level="INFO")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Secrets come from the environment — NEVER hardcode a key in this cell.
# Set it in the shell/kernel BEFORE running:  export GROQ_API_KEY=...
# (Layer A aliasing, Layer B topic labels, and Layer C cards all call the LLM.)
if not os.environ.get("GROQ_API_KEY"):
    print("⚠ GROQ_API_KEY not set — fs.llm will be the MOCK client and the LLM "
          "steps (aliasing / labels / cards) won't run for real. "
          "Export it and re-run this cell.")

BACKEND = "elastic"   # "elastic" (real Elasticsearch + Neo4j stores) or "memory" (offline, reads the parquet)

CORPUS_DIR = ROOT / "data" / "foodscholar" / "corpus"
NEL_DIR    = ROOT / "data" / "foodscholar" / "ner"
SNAPSHOT   = ROOT / "data" / "annotated.parquet"

_storage = (
    {"chunk_store": {"backend": "memory"}, "graph_store": {"backend": "memory"}}
    if BACKEND == "memory" else
    {"chunk_store": {"backend": "elastic", "url": "http://localhost:9200",
                     "index": "foodscholar_chunks"},
     "graph_store": {"backend": "neo4j", "url": "bolt://localhost:7687",
                     "user": "neo4j", "password": os.environ.get("NEO4J_PASSWORD", "password")}}
)

cfg = FoodScholarConfig.model_validate({
    "corpus": {"chunks_path": str(CORPUS_DIR), "annotated_snapshot_path": str(SNAPSHOT)},
    # prefix_filter=None admits the OBO terms embedded in foodon.owl
    # (CHEBI/UBERON/ENVO/…) so the non-food facets can project (see §5).
    "ontology": {"foodon_path": str(ROOT / "data" / "foodon.owl"),
                 "cache_path": str(ROOT / "data" / "foodon_cache.parquet"),
                 "prefix_filter": None},
    "llm": {
        "primary": {"provider": "groq", "model": "llama-3.1-8b-instant"},
    },
    # Layer A: 1a+ backbone projection (the default) + LLM aliasing. The
    # link_blocklist keeps generic mentions off the 'food product' umbrella so
    # those nodes are dropped.
    "layer_a": {
        "projection": "backbone",   # 1a+ backbone-first controlled expansion (default)
        "link_blocklist": [
            {"surface": "foods", "ontology_id": "FOODON:00001002"},
            {"surface": "food products", "ontology_id": "FOODON:00001002"},
            {"surface": "food product", "ontology_id": "FOODON:00001002"},
            {"surface": "perishable foods", "ontology_id": "FOODON:00001002"},
            {"surface": "perishable products", "ontology_id": "FOODON:00001002"},
            {"surface": "certain foods", "ontology_id": "FOODON:00001002"},
            {"surface": "specific food", "ontology_id": "FOODON:00001002"},
            {"surface": "real food", "ontology_id": "FOODON:00001002"},
            {"surface": "imported foods", "ontology_id": "FOODON:00001002"},
            {"surface": "competitive foods", "ontology_id": "FOODON:00001002"},
        ],
    },
    "storage": _storage,
})

fs = FoodScholar.from_config(cfg)
print("backend:", BACKEND, "· llm:", fs.llm.model_id)

## 2. Load corpus + annotations

Offline (`memory`): read the annotated parquet (run `scripts/make_annotated_parquet.py`
once to build it — fast, no Elastic). Real stores (`elastic`): ingest corpus + the
pre-computed NEL CSVs.

In [2]:
if BACKEND == "memory":
    if not SNAPSHOT.exists():
        raise SystemExit("Run scripts/make_annotated_parquet.py first to build "
                         f"{SNAPSHOT}.")
    n = fs.load_chunks(str(SNAPSHOT))
    print(f"loaded {n} chunks from {SNAPSHOT.name}")
else:
    fs.init()
    # Self-heal a stale/auto-created index. If `chunk_id` isn't a `keyword`, the
    # index was created by Elasticsearch's dynamic mapping (chunk_id -> text),
    # NOT by FoodScholar's _create_index — so embed()/scan() can't sort on it
    # and `embedding` would map as float instead of dense_vector (breaking kNN).
    # init() is create-if-missing and won't repair it, so recreate with the real
    # mapping. Safe: the corpus is re-ingested right below, nothing is lost.
    _idx = fs.chunk_store.index
    _cid = (fs.chunk_store._es.indices.get_mapping(index=_idx)[_idx]["mappings"]
            .get("properties", {}).get("chunk_id", {}))
    if _cid.get("type") != "keyword":
        print(f"⚠ {_idx}.chunk_id is {_cid.get('type')!r}, not 'keyword' — "
              "recreating the index with the correct mapping")
        fs.chunk_store.recreate()
    meta = fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR, ignore_source_types=["abstract"])
    print(meta)

HEAD http://localhost:9200/foodscholar_chunks [status:200 duration:0.006s]
HEAD http://localhost:9200/foodscholar_chunks_entities [status:200 duration:0.003s]
Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT shelf_id IF NOT EXISTS FOR (e:Shelf) REQUIRE (e.shelf_id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT shelf_id FOR (e:Shelf) REQUIRE (e.shelf_id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT shelf_id IF NOT EXISTS FOR (s:Shelf) REQUIRE s.shelf_id IS UNIQUE'
Received n

None


## 3. Embed chunks

Needed by Layer B Pass 1 (similarity) and kNN search. A `memory` backend uses a
deterministic **mock** embedder (instant; Pass-1 themes are then non-semantic). A
real backend uses BGE-base.

In [ ]:
meta = fs.embed()
print(meta)

## 4. Build entities

In [ ]:
meta = fs.build_entities()
print(f"entities: {len(fs.entities)}")

## 5. Build Layer A — 1a+ backbone projection + aliasing

`build_layer_a` uses the **1a+** method (`projection="backbone"`, the default):
pick the facet root's supported children as a backbone, then expand each down the
real FoodOn tiers — **collapsing** single-child filing tiers, placing every node
under a **single parent**, capping fan-out, and **pruning** empty dead-ends. Then
the **aliasing pass** gives jargon shelves a human `display_label`. Faithful —
FoodOn ids / is-a edges / original labels intact.

**All facets, not just foods.** With `prefix_filter: None` (admits the OBO terms
embedded in `foodon.owl`) and the **OBO-prefix facet routing** in
`layer_a.facet.PREFIX_TO_FACET`, non-FOODON entities (CHEBI/CDNO/ONS→**nutrients**,
UBERON/MONDO/PATO→**health**, ENVO/GAZ→**sustainability**) project onto real
hierarchies. The per-facet shelf counts below should show nutrients/health/
sustainability populated, not stub roots. (`allergies`/`dietary_patterns` stay at a
stub root unless the corpus links allergen / dietary-pattern entities.)

In [ ]:
meta = fs.build_layer_a()
shelves = fs.graph_store.list_shelves()
by_facet = {}
for s in shelves:
    by_facet[s.facet] = by_facet.get(s.facet, 0) + 1
n_aliased = sum(1 for s in shelves if s.display_label)
print(f"{len(shelves)} shelves · by facet {by_facet} · {n_aliased} aliased")

## 6. Attach chunks to shelves

In [ ]:
meta = fs.attach()
print(meta)

## 7. Build Layer B — themes (BERTopic or Leiden, LLM topics)

Theme discovery backend is selectable via `cfg.layer_b.algorithm`:

- **`"leiden"`** (default): dual-pass — Pass 1 embedding-similarity graph + Leiden, Pass 2 FoodOn
  entity-relatedness + Leiden, then **merge**.
- **`"bertopic"`**: Pass 1 clusters chunk **embeddings directly** with BERTopic (no similarity graph).
  `bertopic.scope` = `"direct"` (shelf's own chunks) or `"subtree"` (own + all descendants');
  `bertopic.clusterer` = `"hdbscan"` (density, auto count, drops outliers) or `"kmeans"`
  (passthrough + matched/auto-K, full coverage). Pass 2 (relatedness) always uses Leiden.

Topic labels are **LLM-generated** (`labeling="llm"`). Use a real backend/embedder for meaningful
clusters (the mock embedder is non-semantic).

In [ ]:
# --- Layer B theme discovery: BERTopic (our tuned baseline) -------------------
fs.config.layer_b.algorithm = "bertopic"           # "bertopic" | "leiden" (Leiden stays available)
fs.config.layer_b.bertopic.scope = "direct"        # "direct" (own chunks) | "subtree" (+ descendants)
fs.config.layer_b.bertopic.clusterer = "hdbscan"   # "hdbscan" (auto count) | "kmeans" (matched/auto-K)
fs.config.layer_b.bertopic.min_topic_size = 15     # min chunks per theme (parallels Leiden min_community_size)
# fs.config.layer_b.bertopic.n_clusters = None     # kmeans only; None → auto sqrt(n/2) clamped [2,12]

fs.config.layer_b.pass1_mode = "per_shelf"         # per-shelf discovery (not global)
fs.config.layer_b.labeling.strategy = "llm"        # LLM-generated topic labels

# Build themes for EVERY facet that Layer A populated (foods + the OBO-projected
# nutrients / health / sustainability). Facets with only a stub root are skipped.
FACETS = [f for f in fs.config.layer_a.facets
          if any(s.facet == f and s.parent_shelf_id is not None
                 for s in fs.graph_store.list_shelves())]
print("theming facets:", FACETS)
for f in FACETS:
    art = fs.build_layer_b(facet=f, dry_run=False)
    print(f"  {f:16} themes: {art.n_themes_total} · by pass {art.n_themes_by_pass} · "
          f"shelves themed {art.n_shelves_themed}")

## 8. Interactive Layer A tree

Render the constructed Layer A foods tree to a self-contained HTML file and show it
inline. Left pane: the full shelf hierarchy with a `Chunks: total (D: direct |
L: lifted)` badge and a `[theme count]` per node (sub-threshold shelves greyed).
Click a shelf to see its Layer B topics on the right, grouped by **origin** —
Merged / Similarity / Relatedness — with a per-origin filter.

In [ ]:
from IPython.display import IFrame

# Full graph across ALL facets (foods + nutrients/health/sustainability). Left pane:
# the shelf hierarchy per facet with a Chunks + [theme count] badge per node; click a
# shelf to see its Layer B topics grouped by origin AND its Layer C card on the right.
# Self-contained HTML in data/viz/. (Pass a single facet name for a focused tree.)
out = fs.viz.layer_a_tree(facet=None).render(
    "tree", output=ROOT / "data" / "viz" / "full_graph_all_facets.html"
)
print("wrote", out)

# IFrame src is relative to this notebook (in notebooks/), hence the ../.
IFrame(src="../data/viz/full_graph_all_facets.html", width="100%", height=700)

## 9. Build Layer C — theme cards (extractive → LLM → embedded Card)

For every Layer B theme: compress its chunks with a cheap **extractive** method (LexRank, map-reduce
when large), refine with the **LLM** (Llama 8B) into a `Card`, **embed** `title + summary`, and persist to
**both** Neo4j and the Elastic `foodscholar_cards` index. One LLM call per theme.

> This is the last pipeline stage. `fs.build()` runs the whole arc in one call:
> `embed → build_entities → build_layer_a → attach → build_layer_b → build_layer_c`.
> The cells above run them individually; this cell adds Layer C on top of the Layer B you just built.

In [ ]:
# extractive Stage-1 method (lexrank won the benchmark); model stamped on each card
fs.config.layer_c.stage1_method = "lexrank"      # lexrank | lsa | luhn | textrank | nltk_freq

# One card per theme, for every facet that got themed in §7. dry_run=True to preview.
for f in FACETS:
    report = fs.build_layer_c(facet=f)
    print(f"  {f:16} {report}")

## 10. Search the cards by vector

`fs.search_cards(text, k=...)` embeds the query, runs kNN over the cards index, and returns the nearest
cards. (Thin retrieval helper — full `query()` with answer synthesis is still deferred.)

In [4]:
for q in ["calcium and vitamin D for bone health",
          "whole grains and dietary fiber"]:
    hits = fs.search_cards(q, k=3)
    print(f"\nquery: {q!r} → {len(hits)} card(s)")
    for c in hits:
        print(f"  [{c.target_id}] {c.title}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.86it/s]
POST http://localhost:9200/foodscholar_cards/_search [status:200 duration:0.020s]
POST http://localhost:9200/foodscholar_cards/_mget [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_cards/_search [status:200 duration:0.006s]



query: 'calcium and vitamin D for bone health' → 3 card(s)
  [foods/foodon_foodon_00001257/milk_and_bone_health_g230] Milk and Dairy for Bone Health
  [foods/foodon_foodon_03315150/calcium_and_vitamin_d_intake_g11] Calcium and Vitamin D Intake Essentials
  [foods/foodon_foodon_03305085/dairy_and_calcium_sources_g131] Dairy Foods are Top Calcium Sources


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.31it/s]
POST http://localhost:9200/foodscholar_cards/_search [status:200 duration:0.006s]
POST http://localhost:9200/foodscholar_cards/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_cards/_search [status:200 duration:0.004s]



query: 'whole grains and dietary fiber' → 3 card(s)
  [foods/foodon_foodon_00001093/types_of_fiber_rich_grains_g239] Fiber-Rich Grains: Choosing the Right Options
  [foods/foodon_foodon_03316531/dietary_fiber_and_whole_foods_r53] Dietary Fiber from Whole Foods is Important
  [foods/foodon_foodon_00002351/whole_grain_food_choices_g161] Whole Grain Food Choices


In [ ]:
fs.export_graphml(ROOT / "data" / "foodscholar_graph.graphml")

PosixPath('/mnt/workspaces/wisefood/foodscholar-lib/data/foodscholar_graph.graphml')